To run this notebook, be sure that "train.csv" is also in the same folder.

In [ ]:
import pandas as pd
import joblib
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from model_training import (
    dummy_regressor_baseline,
    get_default_krr,
    get_default_rf,
    get_default_xgb,
    reg_cv
)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
df_train = pd.read_csv('train.csv')
target_feature = "outputs.pbe.bandgap"
X_train = df_train.drop(columns=[target_feature,'MOFname'])
y_train = df_train[target_feature]

In [ ]:
dummyregressor_mean, dummyregressor_median  = dummy_regressor_baseline(X_train, y_train)
joblib.dump(dummyregressor_mean, 'dummyregressor_mean.pkl')
joblib.dump(dummyregressor_median, 'dummyregressor_median.pkl')

In [ ]:
krr_model_default = get_default_krr(X_train, y_train)
rf_model_default = get_default_rf(X_train, y_train, random_seed=10)
xgb_model_default = get_default_xgb(X_train, y_train, random_seed=10)
joblib.dump(krr_model_default, 'krr_model_default.pkl')
joblib.dump(rf_model_default, 'rf_model_default.pkl')
joblib.dump(xgb_model_default, 'xgb_model_default.pkl')

In [ ]:
krr_param_grid = {
                'scaler': [MinMaxScaler(), StandardScaler(),RobustScaler()], # test different scaling methods
                'krr__alpha': np.logspace(-3,0,10),
                'krr__gamma': np.logspace(-3,3,19)
            }
krr_param_grid = {
                'scaler': [MinMaxScaler()], # test different scaling methods
                'krr__alpha': [0.1],
                'krr__gamma': [0.1]
            }
scorer = 'neg_mean_absolute_error'
krr_search, krr_model = reg_cv(
    model_type = "krr",
    search_type = "gridsearch",
    param = krr_param_grid,
    cv = 3,
    scorer = scorer,
    X_train = X_train,
    y_train = y_train,
    random_seed = 10)
joblib.dump(krr_model, 'krr_best_model.pkl')

In [ ]:
random_forest_param_grid = {
                    'n_estimators': [50, 100, 200, 300, 500, 700],
                    'max_depth': [5, 10, 15, 20, 25, 30],
                    'max_features': [0.5, 0.6, 0.7]
            }
random_forest_param_grid = {
                    'n_estimators': [50],
                    'max_depth': [5],
                    'max_features': [0.5]
            }
cv = KFold(n_splits=3, shuffle=True, random_state=10)
scorer = 'neg_mean_absolute_error'
rf_search, rf_model = reg_cv(
    model_type = "rf",
    search_type = "gridsearch",
    param = random_forest_param_grid,
    cv = cv,
    scorer = scorer,
    X_train = X_train,
    y_train = y_train,
    random_seed = 10)
joblib.dump(rf_model, 'rf_best_model.pkl')

In [ ]:
xgboost_param_grid = {
                    'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
                    'n_estimators': [100, 200, 400, 600, 800, 1000],
                    'max_depth': [3, 4, 5, 6],
                    'min_child_weight': [1, 3, 5],
                    'subsample': [0.6,0.8,1.0],
                    'colsample_bytree': [0.6, 0.8, 1.0]
            }
xgboost_param_grid = {
                    'learning_rate': [0.2],
                    'n_estimators': [100]
            }
cv = KFold(n_splits=3, shuffle=True, random_state=10)
scorer = 'neg_mean_absolute_error'
xgb_search, xgb_model = reg_cv(
    model_type = "xgb",
    search_type = "gridsearch",
    param = xgboost_param_grid,
    cv = cv,
    scorer = scorer,
    X_train = X_train,
    y_train = y_train,
    random_seed = 10)
joblib.dump(xgb_model, 'xgb_best_model.pkl')